In [3]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.applications import EfficientNetV2B0 as BaseModelClass
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: 정규화 완전 제거)
# ==========================================
# EfficientNetV2는 내부에서 자체 스케일링을 수행하므로 [0, 255] 원본을 그대로 줍니다.
train_datagen = ImageDataGenerator(
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 원본 그대로 (Augmentation 없음)
test_datagen = ImageDataGenerator()

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# EfficientNetV2B0 권장 해상도(224x224)로 손실 없이 리사이징
resized = Resizing(224, 224, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "EfficientNetV2B0_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 EfficientNetV2B0 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 EfficientNetV2B0 최종 성능 리포트 (Test 정확도: 0.9263)
              precision    recall  f1-score   support

      Faulty      0.961     0.888     0.923      2000
      Normal      0.896     0.964     0.929      2000

    accuracy                          0.926      4000
   macro avg      0.929     0.926     0.926      4000
weighted avg      0.929     0.926     0.926      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1777          223
True_Normal           72         1928


0

In [4]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 EfficientNetV2B1 모듈 임포트
from tensorflow.keras.applications import EfficientNetV2B1 as BaseModelClass 
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 뒤섞인 데이터프레임의 인덱스를 완전히 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: 정규화 완전 제거)
# ==========================================
# EfficientNetV2는 내부에서 자체 스케일링을 수행하므로 [0, 255] 원본을 그대로 전달
train_datagen = ImageDataGenerator(
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강(Augmentation) 없이 원본 그대로 사용
test_datagen = ImageDataGenerator()

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] EfficientNetV2B1의 권장 해상도(240x240)로 손실 없이 리사이징
resized = Resizing(240, 240, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(240, 240, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
# 🔥 [수정됨] 모델 저장 명칭을 최적화 버전으로 변경
model_name = "EfficientNetV2B1_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류 (0 또는 1)
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 EfficientNetV2B1 이진 분류 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 EfficientNetV2B1 이진 분류 최종 성능 리포트 (Test 정확도: 0.9243)
              precision    recall  f1-score   support

      Faulty      0.957     0.888     0.921      2000
      Normal      0.896     0.960     0.927      2000

    accuracy                          0.924      4000
   macro avg      0.926     0.924     0.924      4000
weighted avg      0.926     0.924     0.924      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1777          223
True_Normal           80         1920


0

In [5]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 EfficientNetV2B2 모듈 임포트
from tensorflow.keras.applications import EfficientNetV2B2 as BaseModelClass 
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: 정규화 완전 제거)
# ==========================================
# EfficientNetV2는 내부에서 자체 스케일링을 수행하므로 [0, 255] 원본을 그대로 전달
train_datagen = ImageDataGenerator(
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강(Augmentation) 없이 원본 그대로 사용
test_datagen = ImageDataGenerator()

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] EfficientNetV2B2의 권장 해상도(260x260)로 손실 없이 리사이징
resized = Resizing(260, 260, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(260, 260, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "EfficientNetV2B2_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 EfficientNetV2B2 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 EfficientNetV2B2 최종 성능 리포트 (Test 정확도: 0.9230)
              precision    recall  f1-score   support

      Faulty      0.945     0.898     0.921      2000
      Normal      0.903     0.948     0.925      2000

    accuracy                          0.923      4000
   macro avg      0.924     0.923     0.923      4000
weighted avg      0.924     0.923     0.923      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1797          203
True_Normal          105         1895


0

In [6]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 EfficientNetB0 (V1) 모듈 임포트
from tensorflow.keras.applications import EfficientNetB0 as BaseModelClass 
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: 정규화 완전 제거)
# ==========================================
# EfficientNet V1 시리즈는 내부에서 자체 스케일링을 수행하므로 원본 [0, 255] 데이터를 그대로 사용
train_datagen = ImageDataGenerator(
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강 없이 원본 그대로 사용
test_datagen = ImageDataGenerator()

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] EfficientNetB0의 권장 해상도(224x224)로 손실 없이 리사이징
resized = Resizing(224, 224, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "EfficientNetB0_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 EfficientNetB0 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 EfficientNetB0 최종 성능 리포트 (Test 정확도: 0.9350)
              precision    recall  f1-score   support

      Faulty      0.964     0.904     0.933      2000
      Normal      0.910     0.966     0.937      2000

    accuracy                          0.935      4000
   macro avg      0.937     0.935     0.935      4000
weighted avg      0.937     0.935     0.935      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1808          192
True_Normal           68         1932


0

In [7]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 EfficientNetB1 (V1) 모듈 임포트
from tensorflow.keras.applications import EfficientNetB1 as BaseModelClass 
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: 정규화 완전 제거)
# ==========================================
# EfficientNet V1 시리즈는 내부에서 자체 스케일링을 수행하므로 원본 [0, 255] 데이터를 그대로 사용
train_datagen = ImageDataGenerator(
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강 없이 원본 그대로 사용
test_datagen = ImageDataGenerator()

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] EfficientNetB1의 권장 해상도(240x240)로 손실 없이 리사이징
resized = Resizing(240, 240, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(240, 240, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "EfficientNetB1_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 EfficientNetB1 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 EfficientNetB1 최종 성능 리포트 (Test 정확도: 0.9380)
              precision    recall  f1-score   support

      Faulty      0.969     0.905     0.936      2000
      Normal      0.911     0.971     0.940      2000

    accuracy                          0.938      4000
   macro avg      0.940     0.938     0.938      4000
weighted avg      0.940     0.938     0.938      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1810          190
True_Normal           58         1942


0

In [8]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 EfficientNetB2 (V1) 모듈 임포트
from tensorflow.keras.applications import EfficientNetB2 as BaseModelClass 
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: 정규화 완전 제거)
# ==========================================
# EfficientNet V1 시리즈는 내부에서 자체 스케일링을 수행하므로 원본 [0, 255] 데이터를 그대로 사용
train_datagen = ImageDataGenerator(
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강 없이 원본 그대로 사용
test_datagen = ImageDataGenerator()

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] EfficientNetB2의 권장 해상도(260x260)로 손실 없이 리사이징
resized = Resizing(260, 260, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(260, 260, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "EfficientNetB2_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 EfficientNetB2 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 EfficientNetB2 최종 성능 리포트 (Test 정확도: 0.9437)
              precision    recall  f1-score   support

      Faulty      0.966     0.920     0.942      2000
      Normal      0.924     0.968     0.945      2000

    accuracy                          0.944      4000
   macro avg      0.945     0.944     0.944      4000
weighted avg      0.945     0.944     0.944      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1840          160
True_Normal           65         1935


0

In [9]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 MobileNet V1 모듈 임포트
from tensorflow.keras.applications import MobileNet as BaseModelClass 
from tensorflow.keras.applications.mobilenet import preprocess_input
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: MobileNet 전용 전처리)
# ==========================================
# MobileNet은 [-1, 1] 범위 입력을 권장하므로 공식 전처리 함수를 사용
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] MobileNet V1 권장 해상도(224x224)로 손실 없이 리사이징
resized = Resizing(224, 224, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "MobileNetV1_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 MobileNet V1 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 MobileNet V1 최종 성능 리포트 (Test 정확도: 0.9367)
              precision    recall  f1-score   support

      Faulty      0.949     0.923     0.936      2000
      Normal      0.925     0.950     0.938      2000

    accuracy                          0.937      4000
   macro avg      0.937     0.937     0.937      4000
weighted avg      0.937     0.937     0.937      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1847          153
True_Normal          100         1900


0

In [10]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 MobileNetV2 모듈 임포트 및 공식 전처리 함수
from tensorflow.keras.applications import MobileNetV2 as BaseModelClass 
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: MobileNetV2 전용 전처리)
# ==========================================
# MobileNetV2는 공식 preprocess_input ([-1, 1] 스케일링) 사용을 강력 권장
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] MobileNetV2의 권장 해상도(224x224)로 손실 없이 리사이징
resized = Resizing(224, 224, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "MobileNetV2_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 MobileNetV2 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 MobileNetV2 최종 성능 리포트 (Test 정확도: 0.9335)
              precision    recall  f1-score   support

      Faulty      0.947     0.918     0.932      2000
      Normal      0.921     0.949     0.934      2000

    accuracy                          0.933      4000
   macro avg      0.934     0.933     0.933      4000
weighted avg      0.934     0.933     0.933      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1837          163
True_Normal          103         1897


0

In [11]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 MobileNetV3Small 모듈 임포트
from tensorflow.keras.applications import MobileNetV3Small as BaseModelClass 
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: 정규화 완전 제거)
# ==========================================
# MobileNetV3는 모델 내부에 자체 스케일링 레이어가 있으므로 [0, 255] 원본을 그대로 전달
train_datagen = ImageDataGenerator(
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강(Augmentation) 없이 원본 그대로 사용
test_datagen = ImageDataGenerator()

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] MobileNetV3의 권장 해상도(224x224)로 손실 없이 리사이징
resized = Resizing(224, 224, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "MobileNetV3Small_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 MobileNet V3 Small 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 MobileNet V3 Small 최종 성능 리포트 (Test 정확도: 0.9440)
              precision    recall  f1-score   support

      Faulty      0.957     0.930     0.943      2000
      Normal      0.932     0.958     0.945      2000

    accuracy                          0.944      4000
   macro avg      0.944     0.944     0.944      4000
weighted avg      0.944     0.944     0.944      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1860          140
True_Normal           84         1916


0

In [13]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 MobileNetV3Large 모듈 임포트
from tensorflow.keras.applications import MobileNetV3Large as BaseModelClass 
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: 정규화 완전 제거)
# ==========================================
# MobileNetV3는 모델 내부에 자체 스케일링 레이어가 있으므로 [0, 255] 원본을 그대로 전달
train_datagen = ImageDataGenerator(
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강(Augmentation) 없이 원본 그대로 사용
test_datagen = ImageDataGenerator()

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] MobileNetV3의 권장 해상도(224x224)로 손실 없이 리사이징
resized = Resizing(224, 224, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "MobileNetV3Large_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 MobileNet V3 Large 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 MobileNet V3 Large 최종 성능 리포트 (Test 정확도: 0.9410)
              precision    recall  f1-score   support

      Faulty      0.955     0.926     0.940      2000
      Normal      0.928     0.956     0.942      2000

    accuracy                          0.941      4000
   macro avg      0.941     0.941     0.941      4000
weighted avg      0.941     0.941     0.941      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1852          148
True_Normal           88         1912


0

In [16]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 DenseNet121 모듈 및 공식 전처리 함수 임포트
from tensorflow.keras.applications import DenseNet121 as BaseModelClass 
from tensorflow.keras.applications.densenet import preprocess_input
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 수정: RGB 모드 적용)
# ==========================================
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input, # 이제 정상 작동합니다.
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='rgb',   # 🔥 흑백 이미지여도 3채널로 자동 복제 로드
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='rgb',   # 🔥 흑백 이미지여도 3채널로 자동 복제 로드
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 수정: 입력 채널 및 파이프라인 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 3)) # 🔥 3채널 입력으로 변경

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# DenseNet121 권장 해상도(224x224)로 손실 없이 리사이징
resized = Resizing(224, 224, interpolation="bicubic")(padded)

# 🔥 (삭제됨) Lambda 레이어는 제거. 데이터 제너레이터에서 이미 3채널로 들어오므로 불필요합니다.

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(resized) # 🔥 리사이징된 이미지를 바로 백본에 연결
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "DenseNet121_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 DenseNet121 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 DenseNet121 최종 성능 리포트 (Test 정확도: 0.9333)
              precision    recall  f1-score   support

      Faulty      0.950     0.914     0.932      2000
      Normal      0.918     0.952     0.934      2000

    accuracy                          0.933      4000
   macro avg      0.934     0.933     0.933      4000
weighted avg      0.934     0.933     0.933      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1829          171
True_Normal           96         1904


0

In [17]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 NASNetMobile 모듈 및 공식 전처리 함수 임포트
from tensorflow.keras.applications import NASNetMobile as BaseModelClass 
from tensorflow.keras.applications.nasnet import preprocess_input
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: NASNet 전용 전처리)
# ==========================================
# NASNetMobile의 권장 스케일링([-1, 1] 변환)을 수행하는 공식 함수 적용
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강 없이 전처리만 수행
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] NASNetMobile 권장 해상도(224x224)로 손실 없이 리사이징
resized = Resizing(224, 224, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "NASNetMobile_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 NASNetMobile 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 NASNetMobile 최종 성능 리포트 (Test 정확도: 0.9073)
              precision    recall  f1-score   support

      Faulty      0.929     0.882     0.905      2000
      Normal      0.888     0.932     0.910      2000

    accuracy                          0.907      4000
   macro avg      0.908     0.907     0.907      4000
weighted avg      0.908     0.907     0.907      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1764          236
True_Normal          135         1865


0

In [1]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 ResNet50 모듈 및 공식 전처리 함수 임포트
from tensorflow.keras.applications import ResNet50 as BaseModelClass 
from tensorflow.keras.applications.resnet50 import preprocess_input
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: ResNet50 전용 전처리)
# ==========================================
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='rgb',            # 🔥 [수정 1] preprocess_input을 위해 rgb(3채널)로 변경
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='rgb',            # 🔥 [수정 1] preprocess_input을 위해 rgb(3채널)로 변경
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 3)) # 🔥 [수정 2] 입력 채널을 1에서 3으로 변경

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# ResNet50 권장 해상도(224x224)로 손실 없이 리사이징
resized = Resizing(224, 224, interpolation="bicubic")(padded)

# 🔥 [수정 3] 데이터 제너레이터에서 3채널로 불러왔으므로 Lambda(grayscale_to_rgb) 레이어 제거
base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(resized)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "ResNet50_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 ResNet50 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 ResNet50 최종 성능 리포트 (Test 정확도: 0.9397)
              precision    recall  f1-score   support

      Faulty      0.955     0.923     0.939      2000
      Normal      0.926     0.956     0.941      2000

    accuracy                          0.940      4000
   macro avg      0.940     0.940     0.940      4000
weighted avg      0.940     0.940     0.940      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1847          153
True_Normal           88         1912


0

In [2]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 ResNet50V2 모듈 및 공식 전처리 함수 임포트
from tensorflow.keras.applications import ResNet50V2 as BaseModelClass 
from tensorflow.keras.applications.resnet_v2 import preprocess_input
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: ResNet V2 전용 전처리)
# ==========================================
# ResNet V2는 [-1, 1] 범위로 입력을 정규화하는 전처리 함수를 사용합니다.
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강 없이 전처리만 수행
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] ResNet50V2 권장 해상도(224x224)로 손실 없이 리사이징
resized = Resizing(224, 224, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "ResNet50V2_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 ResNet50V2 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 ResNet50V2 최종 성능 리포트 (Test 정확도: 0.9443)
              precision    recall  f1-score   support

      Faulty      0.954     0.934     0.944      2000
      Normal      0.935     0.955     0.945      2000

    accuracy                          0.944      4000
   macro avg      0.944     0.944     0.944      4000
weighted avg      0.944     0.944     0.944      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1868          132
True_Normal           91         1909


0

In [3]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 Xception 모듈 및 공식 전처리 함수 임포트
from tensorflow.keras.applications import Xception as BaseModelClass 
from tensorflow.keras.applications.xception import preprocess_input
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: Xception 전용 전처리)
# ==========================================
# Xception 모델에 최적화된 스케일링([-1, 1] 변환)을 수행하는 공식 함수 적용
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강 없이 전처리만 수행
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] Xception 권장 해상도(299x299)로 손실 없이 리사이징
resized = Resizing(299, 299, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "Xception_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 Xception 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 Xception 최종 성능 리포트 (Test 정확도: 0.9473)
              precision    recall  f1-score   support

      Faulty      0.957     0.936     0.947      2000
      Normal      0.938     0.958     0.948      2000

    accuracy                          0.947      4000
   macro avg      0.947     0.947     0.947      4000
weighted avg      0.947     0.947     0.947      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1873          127
True_Normal           84         1916


0

In [1]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 EfficientNetV2B3 모듈 임포트
from tensorflow.keras.applications import EfficientNetV2B3 as BaseModelClass 
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: 정규화 완전 제거)
# ==========================================
# EfficientNetV2는 내부에서 자체 스케일링을 수행하므로 [0, 255] 원본을 그대로 전달
train_datagen = ImageDataGenerator(
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강(Augmentation) 없이 원본 그대로 사용
test_datagen = ImageDataGenerator()

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] EfficientNetV2B3의 권장 해상도(300x300)로 손실 없이 리사이징
resized = Resizing(300, 300, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(300, 300, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "EfficientNetV2B3_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 EfficientNetV2B3 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 EfficientNetV2B3 최종 성능 리포트 (Test 정확도: 0.9247)
              precision    recall  f1-score   support

      Faulty      0.968     0.878     0.921      2000
      Normal      0.889     0.971     0.928      2000

    accuracy                          0.925      4000
   macro avg      0.928     0.925     0.925      4000
weighted avg      0.928     0.925     0.925      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1757          243
True_Normal           58         1942


0

In [2]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 EfficientNetV2S (Small) 모듈 임포트
from tensorflow.keras.applications import EfficientNetV2S as BaseModelClass 
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: 정규화 완전 제거)
# ==========================================
# EfficientNetV2는 내부에서 자체 스케일링을 수행하므로 [0, 255] 원본을 그대로 전달
train_datagen = ImageDataGenerator(
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강(Augmentation) 없이 원본 그대로 사용
test_datagen = ImageDataGenerator()

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] EfficientNetV2S의 권장 해상도(384x384)로 손실 없이 리사이징
resized = Resizing(384, 384, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(384, 384, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "EfficientNetV2S_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 EfficientNetV2S 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 EfficientNetV2S 최종 성능 리포트 (Test 정확도: 0.9307)
              precision    recall  f1-score   support

      Faulty      0.966     0.893     0.928      2000
      Normal      0.901     0.969     0.933      2000

    accuracy                          0.931      4000
   macro avg      0.933     0.931     0.931      4000
weighted avg      0.933     0.931     0.931      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1786          214
True_Normal           63         1937


0

In [1]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 오리지널 EfficientNetB3 (V1) 모듈 임포트
from tensorflow.keras.applications import EfficientNetB3 as BaseModelClass 
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: 정규화 완전 제거)
# ==========================================
# EfficientNet V1 시리즈는 내부에서 자체 스케일링을 수행하므로 원본 [0, 255] 데이터를 그대로 사용
train_datagen = ImageDataGenerator(
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강 없이 원본 그대로 사용
test_datagen = ImageDataGenerator()

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] EfficientNetB3의 권장 해상도(300x300)로 손실 없이 리사이징
resized = Resizing(300, 300, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(300, 300, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "EfficientNetB3_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 EfficientNetB3 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 EfficientNetB3 최종 성능 리포트 (Test 정확도: 0.9323)
              precision    recall  f1-score   support

      Faulty      0.958     0.904     0.930      2000
      Normal      0.910     0.960     0.934      2000

    accuracy                          0.932      4000
   macro avg      0.934     0.932     0.932      4000
weighted avg      0.934     0.932     0.932      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1809          191
True_Normal           80         1920


2

In [2]:
import os
# [수정] CUDA 환경의 GPU 인식을 사전에 차단합니다.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" 

import tensorflow as tf

# ==========================================
# 0. 연산 장치(CPU) 강제 할당 설정
# ==========================================
# [수정] Mac(MPS) 및 모든 GPU 디바이스의 가시성을 비활성화하여 CPU 연산을 강제합니다.
tf.config.set_visible_devices([], 'GPU')
print("🖥️ 설정 완료: GPU(MPS)를 비활성화하고 완전히 CPU로만 학습을 진행합니다.")

import gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 오리지널 EfficientNetB4 (V1) 모듈 임포트
from tensorflow.keras.applications import EfficientNetB4 as BaseModelClass 
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: 정규화 완전 제거)
# ==========================================
# EfficientNet V1 시리즈는 내부에서 자체 스케일링을 수행하므로 원본 [0, 255] 데이터를 그대로 사용
train_datagen = ImageDataGenerator(
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강 없이 원본 그대로 사용
test_datagen = ImageDataGenerator()

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] EfficientNetB4의 권장 해상도(380x380)로 손실 없이 리사이징
resized = Resizing(380, 380, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(380, 380, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "EfficientNetB4_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 EfficientNetB4 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 EfficientNetB4 최종 성능 리포트 (Test 정확도: 0.9380)
              precision    recall  f1-score   support

      Faulty      0.962     0.912     0.936      2000
      Normal      0.917     0.964     0.940      2000

    accuracy                          0.938      4000
   macro avg      0.939     0.938     0.938      4000
weighted avg      0.939     0.938     0.938      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1825          175
True_Normal           73         1927


67

In [3]:
import os
# [수정] CUDA 환경의 GPU 인식을 사전에 차단합니다.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" 

import tensorflow as tf

# ==========================================
# 0. 연산 장치(CPU) 강제 할당 설정
# ==========================================
# [수정] Mac(MPS) 및 모든 GPU 디바이스의 가시성을 비활성화하여 CPU 연산을 강제합니다.
tf.config.set_visible_devices([], 'GPU')
print("🖥️ 설정 완료: GPU(MPS)를 비활성화하고 완전히 CPU로만 학습을 진행합니다.")

import gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 ConvNeXtTiny 모듈 임포트
from tensorflow.keras.applications import ConvNeXtTiny as BaseModelClass 
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: 정규화 완전 제거)
# ==========================================
# ConvNeXt는 내부에서 자체 스케일링을 수행하므로 [0, 255] 원본을 그대로 전달
train_datagen = ImageDataGenerator(
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강(Augmentation) 없이 원본 그대로 사용
test_datagen = ImageDataGenerator()

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] ConvNeXtTiny 권장 해상도(224x224)로 손실 없이 리사이징
resized = Resizing(224, 224, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "ConvNeXtTiny_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    # 🔥 ConvNeXt는 BatchNormalization 대신 LayerNormalization을 사용함
    if isinstance(layer, tf.keras.layers.BatchNormalization) or isinstance(layer, LayerNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    # 🔥 ConvNeXt 전용 LayerNormalization 동결 유지
    if isinstance(layer, tf.keras.layers.BatchNormalization) or isinstance(layer, LayerNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 ConvNeXtTiny 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()

🖥️ 설정 완료: GPU(MPS)를 비활성화하고 완전히 CPU로만 학습을 진행합니다.
📊 데이터셋 분할 및 인덱스 초기화 완료!
훈련 데이터: 16000장, 검증 데이터: 4000장
Found 16000 validated image filenames belonging to 2 classes.
Found 4000 validated image filenames belonging to 2 classes.



🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)
Epoch 1/15


2026-06-18 19:31:22.201395: I external/local_xla/xla/service/service.cc:168] XLA service 0x3231c1fd0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-06-18 19:31:22.201406: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1781778682.213401       1 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-06-18 19:31:22.213498: E external/local_xla/xla/stream_executor/stream_executor_internal.h:177] SetPriority unimplemented for this stream.


250/250 [==============================] - ETA: 0s - loss: 0.5490 - accuracy: 0.7531
Epoch 1: val_accuracy improved from -inf to 0.78725, saving model to /Users/gyuminkang/Desktop/solar/output/ConvNeXtTiny_KaggleOptimized_best.weights.h5
250/250 [==============================] - 1215s 5s/step - loss: 0.5490 - accuracy: 0.7531 - val_loss: 0.5090 - val_accuracy: 0.7872
Epoch 2/15
250/250 [==============================] - ETA: 0s - loss: 0.5007 - accuracy: 0.7966
Epoch 2: val_accuracy improved from 0.78725 to 0.80675, saving model to /Users/gyuminkang/Desktop/solar/output/ConvNeXtTiny_KaggleOptimized_best.weights.h5
250/250 [==============================] - 1209s 5s/step - loss: 0.5007 - accuracy: 0.7966 - val_loss: 0.4861 - val_accuracy: 0.8067
Epoch 3/15
250/250 [==============================] - ETA: 0s - loss: 0.4911 - accuracy: 0.8071
Epoch 3: val_accuracy improved from 0.80675 to 0.81950, saving model to /Users/gyuminkang/Desktop/solar/output/ConvNeXtTiny_KaggleOptimized_best.wei


🏋️ Stage 2: 백본 미세 조정 시작
Epoch 1/30
250/250 [==============================] - ETA: 0s - loss: 0.4340 - accuracy: 0.8547
Epoch 1: val_accuracy improved from 0.84225 to 0.86850, saving model to /Users/gyuminkang/Desktop/solar/output/ConvNeXtTiny_KaggleOptimized_best.weights.h5
250/250 [==============================] - 1669s 7s/step - loss: 0.4340 - accuracy: 0.8547 - val_loss: 0.4113 - val_accuracy: 0.8685
Epoch 2/30
250/250 [==============================] - ETA: 0s - loss: 0.3898 - accuracy: 0.8832
Epoch 2: val_accuracy improved from 0.86850 to 0.89750, saving model to /Users/gyuminkang/Desktop/solar/output/ConvNeXtTiny_KaggleOptimized_best.weights.h5
250/250 [==============================] - 1691s 7s/step - loss: 0.3898 - accuracy: 0.8832 - val_loss: 0.3732 - val_accuracy: 0.8975
Epoch 3/30
250/250 [==============================] - ETA: 0s - loss: 0.3579 - accuracy: 0.9071
Epoch 3: val_accuracy improved from 0.89750 to 0.91300, saving model to /Users/gyuminkang/Desktop/solar/outpu


🏋️ Stage 3: 최종 최적화 시작
Epoch 1/30
250/250 [==============================] - ETA: 0s - loss: 0.2112 - accuracy: 0.9970
Epoch 1: val_accuracy improved from 0.93850 to 0.94050, saving model to /Users/gyuminkang/Desktop/solar/output/ConvNeXtTiny_KaggleOptimized_best.weights.h5
250/250 [==============================] - 1670s 7s/step - loss: 0.2112 - accuracy: 0.9970 - val_loss: 0.3057 - val_accuracy: 0.9405
Epoch 2/30
250/250 [==============================] - ETA: 0s - loss: 0.2098 - accuracy: 0.9976
Epoch 2: val_accuracy did not improve from 0.94050
250/250 [==============================] - 1670s 7s/step - loss: 0.2098 - accuracy: 0.9976 - val_loss: 0.3098 - val_accuracy: 0.9377
Epoch 3/30
250/250 [==============================] - ETA: 0s - loss: 0.2089 - accuracy: 0.9980
Epoch 3: val_accuracy did not improve from 0.94050
250/250 [==============================] - 1652s 7s/step - loss: 0.2089 - accuracy: 0.9980 - val_loss: 0.3091 - val_accuracy: 0.9392
Epoch 4/30
250/250 [============

KeyboardInterrupt: 

In [ ]:
import os
# [수정] CUDA 환경의 GPU 인식을 사전에 차단합니다.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" 

import tensorflow as tf

# ==========================================
# 0. 연산 장치(CPU) 강제 할당 설정
# ==========================================
# [수정] Mac(MPS) 및 모든 GPU 디바이스의 가시성을 비활성화하여 CPU 연산을 강제합니다.
tf.config.set_visible_devices([], 'GPU')
print("🖥️ 설정 완료: GPU(MPS)를 비활성화하고 완전히 CPU로만 학습을 진행합니다.")

import gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 ConvNeXtSmall 모듈 임포트
from tensorflow.keras.applications import ConvNeXtSmall as BaseModelClass 
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: 정규화 완전 제거)
# ==========================================
# ConvNeXt는 내부에서 자체 스케일링을 수행하므로 [0, 255] 원본을 그대로 전달
train_datagen = ImageDataGenerator(
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강(Augmentation) 없이 원본 그대로 사용
test_datagen = ImageDataGenerator()

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] ConvNeXtSmall 권장 해상도(224x224)로 손실 없이 리사이징
resized = Resizing(224, 224, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "ConvNeXtSmall_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    # 🔥 ConvNeXt는 BatchNormalization 대신 LayerNormalization을 사용함
    if isinstance(layer, tf.keras.layers.BatchNormalization) or isinstance(layer, LayerNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    # 🔥 ConvNeXt 전용 LayerNormalization 동결 유지
    if isinstance(layer, tf.keras.layers.BatchNormalization) or isinstance(layer, LayerNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 ConvNeXtSmall 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()

In [ ]:
import os
# [수정] CUDA 환경의 GPU 인식을 사전에 차단합니다.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" 

import tensorflow as tf

# ==========================================
# 0. 연산 장치(CPU) 강제 할당 설정
# ==========================================
# [수정] Mac(MPS) 및 모든 GPU 디바이스의 가시성을 비활성화하여 CPU 연산을 강제합니다.
tf.config.set_visible_devices([], 'GPU')
print("🖥️ 설정 완료: GPU(MPS)를 비활성화하고 완전히 CPU로만 학습을 진행합니다.")

import gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 DenseNet169 모듈 및 공식 전처리 함수 임포트
from tensorflow.keras.applications import DenseNet169 as BaseModelClass 
from tensorflow.keras.applications.densenet import preprocess_input
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: DenseNet 전용 전처리)
# ==========================================
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='rgb',             # 🔥 [수정 1] preprocess_input 연산을 위해 rgb(3채널)로 변경
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='rgb',             # 🔥 [수정 1] preprocess_input 연산을 위해 rgb(3채널)로 변경
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 3)) # 🔥 [수정 2] 입력 채널을 1에서 3으로 변경

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# DenseNet169 권장 해상도(224x224)로 손실 없이 리사이징
resized = Resizing(224, 224, interpolation="bicubic")(padded)

# 🔥 [수정 3] 데이터 제너레이터에서 3채널로 불러왔으므로 Lambda 레이어 제거 후 바로 base_model 연결
base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(resized)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "DenseNet169_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 DenseNet169 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()

In [4]:
import os
# [수정] CUDA 환경의 GPU 인식을 사전에 차단합니다.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" 

import tensorflow as tf

# ==========================================
# 0. 연산 장치(CPU) 강제 할당 설정
# ==========================================
# [수정] Mac(MPS) 및 모든 GPU 디바이스의 가시성을 비활성화하여 CPU 연산을 강제합니다.
tf.config.set_visible_devices([], 'GPU')
print("🖥️ 설정 완료: GPU(MPS)를 비활성화하고 완전히 CPU로만 학습을 진행합니다.")

import gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 InceptionV3 모듈 및 공식 전처리 함수 임포트
from tensorflow.keras.applications import InceptionV3 as BaseModelClass 
from tensorflow.keras.applications.inception_v3 import preprocess_input
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: InceptionV3 전용 전처리)
# ==========================================
# InceptionV3에 최적화된 스케일링([-1, 1] 변환)을 수행하는 공식 함수 적용
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강 없이 전처리만 수행
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] InceptionV3 권장 해상도(299x299)로 손실 없이 리사이징
resized = Resizing(299, 299, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "InceptionV3_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 InceptionV3 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 InceptionV3 최종 성능 리포트 (Test 정확도: 0.9297)
              precision    recall  f1-score   support

      Faulty      0.954     0.903     0.928      2000
      Normal      0.908     0.956     0.932      2000

    accuracy                          0.930      4000
   macro avg      0.931     0.930     0.930      4000
weighted avg      0.931     0.930     0.930      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1807          193
True_Normal           88         1912


260

In [5]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 ResNet101 모듈 및 공식 전처리 함수 임포트
from tensorflow.keras.applications import ResNet101 as BaseModelClass 
from tensorflow.keras.applications.resnet import preprocess_input
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: ResNet 전용 전처리)
# ==========================================
# ResNet은 원본 [0, 255] 데이터를 받아 내부적으로 Caffe 스타일 변환을 수행하는 함수 적용
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강 없이 전처리만 수행
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='rgb',             # 🔥 [수정 1] preprocess_input 연산을 위해 rgb(3채널)로 변경
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='rgb',             # 🔥 [수정 1] preprocess_input 연산을 위해 rgb(3채널)로 변경
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 3)) # 🔥 [수정 2] 입력 채널을 1에서 3으로 변경

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# ResNet101 권장 해상도(224x224)로 손실 없이 리사이징
resized = Resizing(224, 224, interpolation="bicubic")(padded)

# 🔥 [수정 3] 데이터 제너레이터에서 3채널로 불러왔으므로 Lambda 복제 레이어 제거 후 바로 base_model 연결
base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(resized) 
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "ResNet101_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 ResNet101 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 ResNet101 최종 성능 리포트 (Test 정확도: 0.9475)
              precision    recall  f1-score   support

      Faulty      0.973     0.921     0.946      2000
      Normal      0.925     0.974     0.949      2000

    accuracy                          0.948      4000
   macro avg      0.949     0.948     0.947      4000
weighted avg      0.949     0.948     0.947      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1842          158
True_Normal           52         1948


260

In [6]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 ResNet101V2 모듈 및 공식 전처리 함수 임포트
from tensorflow.keras.applications import ResNet101V2 as BaseModelClass 
from tensorflow.keras.applications.resnet_v2 import preprocess_input
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: ResNet V2 전용 전처리)
# ==========================================
# ResNet V2는 [-1, 1] 범위로 입력을 정규화하는 공식 전처리 함수를 사용합니다.
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강 없이 전처리만 수행
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='grayscale',       
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 1))

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] ResNet101V2 권장 해상도(224x224)로 손실 없이 리사이징
resized = Resizing(224, 224, interpolation="bicubic")(padded)

# 🔥 [수정] 데이터 유실을 일으키던 Conv2D(relu) 대신, 단순 채널 복제로 1채널을 3채널로 변환
input_adapter = Lambda(lambda img: tf.image.grayscale_to_rgb(img))(resized)

base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(input_adapter)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "ResNet101V2_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (50 레이어 해제)
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-50]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (75 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-75]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 ResNet101V2 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 ResNet101V2 최종 성능 리포트 (Test 정확도: 0.9477)
              precision    recall  f1-score   support

      Faulty      0.961     0.933     0.947      2000
      Normal      0.935     0.963     0.949      2000

    accuracy                          0.948      4000
   macro avg      0.948     0.948     0.948      4000
weighted avg      0.948     0.948     0.948      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1866          134
True_Normal           75         1925


67

In [ ]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 VGG16 모듈 및 공식 전처리 함수 임포트
from tensorflow.keras.applications import VGG16 as BaseModelClass 
from tensorflow.keras.applications.vgg16 import preprocess_input
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

# 데이터프레임 인덱스 완전 초기화
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 핵심 수정: VGG16 전용 전처리)
# ==========================================
# VGG16은 Caffe 스타일(BGR 변환 및 ImageNet 평균 차감)의 전처리를 수행합니다.
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

# 검증 데이터는 데이터 증강 없이 전처리만 수행
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='rgb',             # 🔥 [수정 1] preprocess_input 연산을 위해 rgb(3채널)로 변경
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='rgb',             # 🔥 [수정 1] preprocess_input 연산을 위해 rgb(3채널)로 변경
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 핵심 수정: 무손실 어댑터 및 해상도 최적화)
# ==========================================
inputs = Input(shape=(40, 24, 3)) # 🔥 [수정 2] 입력 채널을 1에서 3으로 변경

# 40x24 이미지를 여백(Padding)을 주어 40x40 정사각형으로 만듦 (비율 왜곡 방지)
padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# VGG16 권장 해상도(224x224)로 손실 없이 리사이징
resized = Resizing(224, 224, interpolation="bicubic")(padded)

# 🔥 [수정 3] 데이터 제너레이터에서 3채널로 불러왔으므로 Lambda 복제 레이어 제거 후 바로 base_model 연결
base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(resized)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "VGG16_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (10 레이어 해제) - VGG16 맞춤형
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-10]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (15 레이어 해제 + CosineDecay)
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-15]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 VGG16 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()

📊 데이터셋 분할 및 인덱스 초기화 완료!
훈련 데이터: 16000장, 검증 데이터: 4000장
Found 16000 validated image filenames belonging to 2 classes.
Found 4000 validated image filenames belonging to 2 classes.

🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)
Epoch 1/15
  3/250 ━━━━━━━━━━━━━━━━━━━━ 6:20 2s/step - accuracy: 0.5330 - loss: 0.7947

KeyboardInterrupt: 

In [7]:
import os, gc, itertools, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 🔥 [수정] InceptionResNetV2 모듈 및 공식 전처리 함수 임포트
from tensorflow.keras.applications import InceptionResNetV2 as BaseModelClass 
from tensorflow.keras.applications.inception_resnet_v2 import preprocess_input
from IPython.display import clear_output

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

if not os.path.exists(config.OUTPUT_PATH):
    os.makedirs(config.OUTPUT_PATH)

# ==========================================
# 2. 메타데이터 로드 및 이진 분류 레이블 적용
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')

with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))

# [1등 치트키 1] 이진 분류 레이블 맵핑
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

# [1등 치트키 2] 비율 균등 분할 기법(Stratified Split) 적용 (8:2)
train_df, val_df = train_test_split(
    df, 
    train_size=0.8, 
    shuffle=True, 
    random_state=42, 
    stratify=df['anomaly_class']
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("📊 데이터셋 분할 및 인덱스 초기화 완료!")
print(f"훈련 데이터: {len(train_df)}장, 검증 데이터: {len(val_df)}장")

# ==========================================
# 3. 데이터 제너레이터 (🔥 InceptionResNetV2 전용 전처리)
# ==========================================
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,            
    horizontal_flip=True, 
    vertical_flip=False           
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

trainGen = train_datagen.flow_from_dataframe(
    train_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='rgb',             
    class_mode='binary'
)

testGen = test_datagen.flow_from_dataframe(
    val_df, x_col='filepath', y_col='status',
    target_size=(40, 24), batch_size=32, 
    color_mode='rgb',             
    class_mode='binary', shuffle=False
)

# ==========================================
# 4. 모델 아키텍처 설계 (🔥 해상도 최적화: 299x299)
# ==========================================
inputs = Input(shape=(40, 24, 3)) 

padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)

# 🔥 [수정] Inception 계열 모델의 권장 해상도(299x299)로 리사이징
resized = Resizing(299, 299, interpolation="bicubic")(padded)

# 🔥 [수정] input_shape를 299로 맞춤
base_model = BaseModelClass(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
base_model.trainable = False

x = base_model(resized)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)               

outputs = Dense(1, activation="sigmoid")(x) 

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 점진적 대용량 학습 스케줄 실행
# ==========================================
model_name = "InceptionResNetV2_KaggleOptimized"
checkpoint_weights_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_weights_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)
]

# [Stage 1] 분류기 웜업
print("\n🏋️ Stage 1: 분류기 웜업 시작 (정석 이진 분류)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=15, callbacks=callbacks, verbose=1)

# [Stage 2] 미세 조정 (100 레이어 해제) - 🔥 [수정] 레이어가 매우 깊어 100개 해제
print("\n🏋️ Stage 2: 백본 미세 조정 시작")
base_model.trainable = True
for layer in base_model.layers[:-100]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# [Stage 3] 최종 미세 조정 (150 레이어 해제 + CosineDecay) - 🔥 [수정] 150개 해제
print("\n🏋️ Stage 3: 최종 최적화 시작")
for layer in base_model.layers[:-150]: 
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * 250 
model.compile(optimizer=Adam(CosineDecay(5e-5, total_steps)), loss=loss_fn, metrics=["accuracy"])
model.fit(trainGen, steps_per_epoch=250, validation_data=testGen, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가 및 리포트 출력
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

# 시그모이드 출력이므로 0.5 기준으로 클래스 분류
probs = model.predict(testGen, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()
acc = accuracy_score(testGen.classes, y_pred)

print(f"\n📊 최적화된 InceptionResNetV2 최종 성능 리포트 (Test 정확도: {acc:.4f})")
print(classification_report(testGen.classes, y_pred, target_names=list(trainGen.class_indices.keys()), digits=3))

cm = confusion_matrix(testGen.classes, y_pred)
print("\n🧩 혼동 행렬(Confusion Matrix) 결과:")
print(pd.DataFrame(cm, index=['True_Faulty', 'True_Normal'], columns=['Pred_Faulty', 'Pred_Normal']))

del model, base_model
K.clear_session()
gc.collect()


📊 최적화된 InceptionResNetV2 최종 성능 리포트 (Test 정확도: 0.9380)
              precision    recall  f1-score   support

      Faulty      0.960     0.914     0.937      2000
      Normal      0.918     0.962     0.939      2000

    accuracy                          0.938      4000
   macro avg      0.939     0.938     0.938      4000
weighted avg      0.939     0.938     0.938      4000


🧩 혼동 행렬(Confusion Matrix) 결과:
             Pred_Faulty  Pred_Normal
True_Faulty         1829          171
True_Normal           77         1923


67

In [1]:
import os, gc, json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, Resizing
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay

# 🔥 [수정됨] 버전 충돌이 없는 공식 ResNet50V2 및 전용 전처리 함수 로드
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.applications.resnet_v2 import preprocess_input

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from IPython.display import clear_output

print(f"TensorFlow: {tf.__version__}")
print("✅ 공식 ResNet50V2 (ImageNet 사전학습) 로드 완료")

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

os.makedirs(config.OUTPUT_PATH, exist_ok=True)

# ==========================================
# 2. 메타데이터 로드 및 분할
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')
with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

train_df, val_df = train_test_split(
    df, train_size=0.8, shuffle=True, random_state=42, stratify=df['status']
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"📊 훈련: {len(train_df)}장, 검증: {len(val_df)}장")

# ==========================================
# 3. tf.data 파이프라인
# ==========================================
IMG_SIZE = (40, 24)
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

class_names = sorted(df['status'].unique())
class_to_idx = {name: i for i, name in enumerate(class_names)}
print(f"클래스 매핑: {class_to_idx}")

train_labels = train_df['status'].map(class_to_idx).values.astype(np.float32)
val_labels = val_df['status'].map(class_to_idx).values.astype(np.float32)

def load_and_preprocess(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    img = preprocess_input(img)
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 2, dtype=tf.int32))
    return img, label

train_ds = tf.data.Dataset.from_tensor_slices((train_df['filepath'].values, train_labels))
train_ds = train_ds.shuffle(len(train_df)).map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.map(augment, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_df['filepath'].values, val_labels))
val_ds = val_ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

# ==========================================
# 4. 모델 아키텍처 (🔥 [수정됨] ResNet50V2 + ImageNet)
# ==========================================
inputs = Input(shape=(40, 24, 3))
resized = Resizing(224, 224, interpolation="bicubic")(inputs)

base_model = ResNet50V2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(resized)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 2-Stage Transfer Learning
# ==========================================
model_name = "ResNet50V2_Official"
checkpoint_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")
steps_per_epoch = len(train_df) // BATCH_SIZE

callbacks = [
    ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
]

print("\n🏋️ Stage 1: 분류기 웜업 (백본 동결)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks, verbose=1)

print("\n🏋️ Stage 2: 백본 미세 조정 (언프리징)")
base_model.trainable = True
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * steps_per_epoch
lr_schedule = CosineDecay(1e-4, total_steps, alpha=0.1)
model.compile(optimizer=Adam(learning_rate=lr_schedule), loss=loss_fn, metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

probs = model.predict(val_ds, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()

print(f"\n📊 공식 ResNet50V2 최종 성능 리포트 (Test 정확도: {accuracy_score(val_labels, y_pred):.4f})")
print(classification_report(val_labels, y_pred, target_names=class_names, digits=3))

del model, base_model
K.clear_session()
gc.collect()


📊 공식 ResNet50V2 최종 성능 리포트 (Test 정확도: 0.9425)
              precision    recall  f1-score   support

      Faulty      0.953     0.931     0.942      2000
      Normal      0.933     0.954     0.943      2000

    accuracy                          0.943      4000
   macro avg      0.943     0.943     0.942      4000
weighted avg      0.943     0.943     0.942      4000



0

In [2]:
import os, gc, json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, Resizing
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers.schedules import CosineDecay

# 🔥 [수정됨] 버전 충돌이 없는 공식 ResNet101V2 및 전용 전처리 함수 로드
from tensorflow.keras.applications import ResNet101V2
from tensorflow.keras.applications.resnet_v2 import preprocess_input

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from IPython.display import clear_output

print(f"TensorFlow: {tf.__version__}")
print("✅ 공식 ResNet101V2 (ImageNet 사전학습) 로드 완료")

# ==========================================
# 1. 환경 설정
# ==========================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

os.makedirs(config.OUTPUT_PATH, exist_ok=True)

# ==========================================
# 2. 메타데이터 로드 및 분할
# ==========================================
metadata_path = os.path.join(config.DEFECTS_PATH, 'module_metadata.json')
with open(metadata_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

train_df, val_df = train_test_split(
    df, train_size=0.8, shuffle=True, random_state=42, stratify=df['status']
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"📊 훈련: {len(train_df)}장, 검증: {len(val_df)}장")

# ==========================================
# 3. tf.data 파이프라인
# ==========================================
IMG_SIZE = (40, 24)
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

class_names = sorted(df['status'].unique())
class_to_idx = {name: i for i, name in enumerate(class_names)}
print(f"클래스 매핑: {class_to_idx}")

train_labels = train_df['status'].map(class_to_idx).values.astype(np.float32)
val_labels = val_df['status'].map(class_to_idx).values.astype(np.float32)

def load_and_preprocess(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    img = preprocess_input(img)
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 2, dtype=tf.int32))
    return img, label

train_ds = tf.data.Dataset.from_tensor_slices((train_df['filepath'].values, train_labels))
train_ds = train_ds.shuffle(len(train_df)).map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.map(augment, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_df['filepath'].values, val_labels))
val_ds = val_ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

steps_per_epoch = len(train_df) // BATCH_SIZE

# ==========================================
# 4. 모델 아키텍처 (🔥 [수정됨] ResNet101V2 + ImageNet)
# ==========================================
inputs = Input(shape=(40, 24, 3))
resized = Resizing(224, 224, interpolation="bicubic")(inputs)

base_model = ResNet101V2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model(resized)
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs=inputs, outputs=outputs)
loss_fn = BinaryCrossentropy(label_smoothing=0.1)

# ==========================================
# 5. 2-Stage Transfer Learning
# ==========================================
model_name = "ResNet101V2_Official"
checkpoint_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_best.weights.h5")
final_model_path = os.path.join(config.OUTPUT_PATH, f"{model_name}_final.keras")

callbacks = [
    ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
]

print("\n🏋️ Stage 1: 분류기 웜업 (백본 동결)")
model.compile(optimizer=Adam(1e-3), loss=loss_fn, metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks, verbose=1)

print("\n🏋️ Stage 2: 백본 미세 조정 (언프리징)")
base_model.trainable = True
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

total_steps = 30 * steps_per_epoch
lr_schedule = CosineDecay(1e-4, total_steps, alpha=0.1)
model.compile(optimizer=Adam(learning_rate=lr_schedule), loss=loss_fn, metrics=["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=callbacks, verbose=1)

# ==========================================
# 6. 최종 평가
# ==========================================
model.save(final_model_path)
clear_output(wait=True)

probs = model.predict(val_ds, verbose=0)
y_pred = (probs > 0.5).astype(int).flatten()

print(f"\n📊 공식 ResNet101V2 최종 성능 리포트 (Test 정확도: {accuracy_score(val_labels, y_pred):.4f})")
print(classification_report(val_labels, y_pred, target_names=class_names, digits=3))

del model, base_model
K.clear_session()
gc.collect()

TensorFlow: 2.20.0
✅ 공식 ResNet101V2 (ImageNet 사전학습) 로드 완료
📊 훈련: 16000장, 검증: 4000장
클래스 매핑: {'Faulty': 0, 'Normal': 1}

🏋️ Stage 1: 분류기 웜업 (백본 동결)
Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6789 - loss: 0.6765
Epoch 1: val_accuracy improved from None to 0.81875, saving model to /Users/gyuminkang/Desktop/solar/output/ResNet101V2_Official_best.weights.h5

Epoch 1: finished saving model to /Users/gyuminkang/Desktop/solar/output/ResNet101V2_Official_best.weights.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 520s 2s/step - accuracy: 0.7278 - loss: 0.5856 - val_accuracy: 0.8188 - val_loss: 0.4727
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7744 - loss: 0.5210
Epoch 2: val_accuracy improved from 0.81875 to 0.83400, saving model to /Users/gyuminkang/Desktop/solar/output/ResNet101V2_Official_best.weights.h5

Epoch 2: finished saving model to /Users/gyuminkang/Desktop/solar/output/ResNet101V2_Official_best.weights.h5
250/250 ━━━━━━━━━━━━━━━━━━━━ 517s 2s/step - accuracy

KeyboardInterrupt: 

In [ ]:
import os, gc, json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, ZeroPadding2D, Resizing
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from IPython.display import clear_output

# =========================================================
# 1. 모든 Base Model 및 전용 전처리 함수 임포트
# =========================================================
from tensorflow.keras.applications import (
    EfficientNetB0, EfficientNetB1, EfficientNetB2, EfficientNetB3, EfficientNetB4,
    EfficientNetV2B0, EfficientNetV2B1, EfficientNetV2B2, EfficientNetV2B3, EfficientNetV2S,
    MobileNet, MobileNetV2, MobileNetV3Small, MobileNetV3Large,
    DenseNet121, DenseNet169, NASNetMobile,
    ResNet50, ResNet50V2, ResNet101, ResNet101V2,
    Xception, InceptionV3, ConvNeXtTiny, ConvNeXtSmall, VGG16,
    InceptionResNetV2 # 🔥 [추가] InceptionResNetV2 임포트
)

# Keras 공식 전처리 함수 모음
from tensorflow.keras.applications.mobilenet import preprocess_input as pre_mobilenet
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as pre_mobilenet_v2
from tensorflow.keras.applications.densenet import preprocess_input as pre_densenet
from tensorflow.keras.applications.nasnet import preprocess_input as pre_nasnet
from tensorflow.keras.applications.resnet50 import preprocess_input as pre_resnet50
from tensorflow.keras.applications.resnet_v2 import preprocess_input as pre_resnet_v2
from tensorflow.keras.applications.resnet import preprocess_input as pre_resnet
from tensorflow.keras.applications.xception import preprocess_input as pre_xception
from tensorflow.keras.applications.inception_v3 import preprocess_input as pre_inception_v3
from tensorflow.keras.applications.vgg16 import preprocess_input as pre_vgg16
from tensorflow.keras.applications.inception_resnet_v2 import preprocess_input as pre_inception_resnet_v2 # 🔥 [추가]

# =========================================================
# 2. 모델별 개별 설정 딕셔너리 구성 (클래스, 해상도, 전처리함수)
# =========================================================
model_configs = {
    "EfficientNetB0": (EfficientNetB0, 224, None),
    "EfficientNetB1": (EfficientNetB1, 240, None),
    "EfficientNetB2": (EfficientNetB2, 260, None),
    "EfficientNetB3": (EfficientNetB3, 300, None),
    "EfficientNetB4": (EfficientNetB4, 380, None),
    "EfficientNetV2B0": (EfficientNetV2B0, 224, None),
    "EfficientNetV2B1": (EfficientNetV2B1, 240, None),
    "EfficientNetV2B2": (EfficientNetV2B2, 260, None),
    "EfficientNetV2B3": (EfficientNetV2B3, 300, None),
    "EfficientNetV2S": (EfficientNetV2S, 384, None),
    "MobileNetV1": (MobileNet, 224, pre_mobilenet),
    "MobileNetV2": (MobileNetV2, 224, pre_mobilenet_v2),
    "MobileNetV3Small": (MobileNetV3Small, 224, None),
    "MobileNetV3Large": (MobileNetV3Large, 224, None),
    "DenseNet121": (DenseNet121, 224, pre_densenet),
    "DenseNet169": (DenseNet169, 224, pre_densenet),
    "NASNetMobile": (NASNetMobile, 224, pre_nasnet),
    "ResNet50": (ResNet50, 224, pre_resnet50),
    "ResNet50V2": (ResNet50V2, 224, pre_resnet_v2),
    "ResNet101": (ResNet101, 224, pre_resnet),
    "ResNet101V2": (ResNet101V2, 224, pre_resnet_v2),
    "Xception": (Xception, 299, pre_xception),
    "InceptionV3": (InceptionV3, 299, pre_inception_v3),
    "InceptionResNetV2": (InceptionResNetV2, 299, pre_inception_resnet_v2), # 🔥 [추가] 299 해상도 매핑
    "ConvNeXtTiny": (ConvNeXtTiny, 224, None),
    "ConvNeXtSmall": (ConvNeXtSmall, 224, None),
    "VGG16": (VGG16, 224, pre_vgg16)
}

# =========================================================
# 3. 데이터프레임 복원
# =========================================================
class config:
    DEFECTS_PATH = "/Users/gyuminkang/Desktop/solar/InfraredSolarModules"
    OUTPUT_PATH = "/Users/gyuminkang/Desktop/solar/output"

with open(os.path.join(config.DEFECTS_PATH, 'module_metadata.json'), 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data, orient='index')
df['filepath'] = df['image_filepath'].apply(lambda x: os.path.join(config.DEFECTS_PATH, x))
df['status'] = df['anomaly_class'].apply(lambda x: 'Normal' if x == 'No-Anomaly' else 'Faulty')

_, val_df = train_test_split(df, train_size=0.8, shuffle=True, random_state=42, stratify=df['anomaly_class'])
val_df = val_df.reset_index(drop=True)

# =========================================================
# 4. 순위 집계 로직 실행 (동적 파이프라인)
# =========================================================
results = []
print(f"🚀 총 {len(model_configs)}개의 모델 정밀 평가를 시작합니다...\n")

for model_prefix, (BaseModelClass, img_size, preproc_fn) in model_configs.items():
    
    opt_weights = os.path.join(config.OUTPUT_PATH, f"{model_prefix}_KaggleOptimized_best.weights.h5")
    old_weights = os.path.join(config.OUTPUT_PATH, f"{model_prefix}_TrueBinary_best.weights.h5")
    
    if os.path.exists(opt_weights):
        weights_path = opt_weights
        version = "(Optimized)"
    elif os.path.exists(old_weights):
        weights_path = old_weights
        version = "(Old Ver)"
    else:
        print(f"⚠️ 건너뜀: {model_prefix} 가중치 파일을 찾을 수 없습니다.")
        continue
        
    print(f"🔄 평가 중: {model_prefix} {version} [해상도: {img_size}] ...", end=" ")
    
    try:
        test_datagen = ImageDataGenerator(preprocessing_function=preproc_fn)
        testGen = test_datagen.flow_from_dataframe(
            val_df, x_col='filepath', y_col='status',
            target_size=(40, 24), batch_size=32, 
            color_mode='rgb',          # 🔥 [수정] 평가 루프 채널 에러 방지를 위해 강제 3채널(rgb) 로드
            class_mode='binary', shuffle=False
        )
        
        y_true = testGen.classes
        
        # 🔥 [수정] Lambda 복제 제거 및 Input 채널을 3으로 직접 매핑
        inputs = Input(shape=(40, 24, 3))
        padded = ZeroPadding2D(padding=((0, 0), (8, 8)))(inputs)
        resized = Resizing(img_size, img_size, interpolation="bicubic")(padded)
        
        base_model = BaseModelClass(weights=None, include_top=False, input_shape=(img_size, img_size, 3))
        
        x = base_model(resized)
        x = GlobalAveragePooling2D()(x)
        x = Dense(512, activation="relu")(x)
        x = Dropout(0.4)(x)
        outputs = Dense(1, activation="sigmoid")(x)
        model = Model(inputs=inputs, outputs=outputs)
        
        model.load_weights(weights_path)
        
        probs = model.predict(testGen, verbose=0)
        y_pred = (probs > 0.5).astype(int).flatten()
        
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average='weighted')
        
        results.append({
            "Model": model_prefix,
            "Accuracy": acc,
            "F1-Score": f1,
            "Resolution": img_size,
            "Version": version.replace("(", "").replace(")", "")
        })
        print(f"완료! (Acc: {acc:.4f})")
        
    except Exception as e:
        print(f"❌ 에러 발생: {e}")
        
    finally:
        del model, base_model, testGen, test_datagen
        K.clear_session()
        gc.collect()

# =========================================================
# 5. 최종 리더보드 출력
# =========================================================
if len(results) > 0:
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(by=['Accuracy', 'F1-Score'], ascending=[False, False]).reset_index(drop=True)
    results_df.index += 1
    
    print("\n" + "="*70)
    print("🏆 열화상 태양광 패널 모델 최종 리더보드 (다중 해상도 지원) 🏆")
    print("="*70)
    print(results_df.to_markdown())
    
    csv_path = os.path.join(config.OUTPUT_PATH, "final_optimized_leaderboard.csv")
    results_df.to_csv(csv_path)
    print(f"\n💾 리더보드가 저장되었습니다: {csv_path}")
else:
    print("\n❌ 평가할 가중치 파일을 찾지 못했습니다. 경로와 파일명을 다시 확인해주세요.")